# Lab 5B: Alpha-Beta Pruning and Move Ordering

## Learning goals
By the end of this lab, students can:
- explain why alpha-beta pruning returns the same minimax decision without evaluating every leaf;
- trace the changing alpha and beta bounds in a game tree;
- see exactly which subtrees are pruned;
- compare minimax, random-order alpha-beta, and best-order alpha-beta on random trees;
- connect move ordering to the Chapter 5 complexity discussion.

Source note: The first tree mirrors the Chapter 5 two-ply minimax / alpha-beta example, then extends it with experiments.

## Chapter 5 notes for this lab

Alpha-beta pruning keeps two bounds along the current depth-first path:

- **alpha**: the best value found so far for MAX, so MAX can guarantee at least this much;
- **beta**: the best value found so far for MIN, so MIN can hold MAX to at most this much.

A subtree can be pruned when the current node is already worse than an alternative available higher in the tree. With perfect move ordering, alpha-beta can examine about \(O(b^{m/2})\) nodes rather than \(O(b^m)\), which is why move ordering matters.

In [ ]:
# Run this cell first.
# %pip install matplotlib ipywidgets

import math
import random
from statistics import mean
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Static first/last frames will still work.")


def make_stepper(steps, draw_func, info_func=None, title="Search stepper"):
    if not steps:
        print("No steps to display.")
        return
    if WIDGETS_AVAILABLE:
        slider = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, description="step")
        play = widgets.Play(value=0, min=0, max=len(steps)-1, step=1, interval=700)
        widgets.jslink((play, "value"), (slider, "value"))
        out = widgets.Output()
        def render(change=None):
            with out:
                clear_output(wait=True)
                display(HTML(f"<h3>{title}</h3>"))
                if info_func:
                    display(HTML(info_func(steps[slider.value], slider.value)))
                draw_func(steps[slider.value], slider.value)
                plt.show()
        slider.observe(render, names="value")
        display(widgets.VBox([widgets.HBox([play, slider]), out]))
        render()
    else:
        for i in [0, len(steps)-1]:
            if info_func:
                display(HTML(info_func(steps[i], i)))
            draw_func(steps[i], i)
            plt.show()


def fmt_bound(x):
    if x == math.inf:
        return "+∞"
    if x == -math.inf:
        return "-∞"
    return f"{x:g}"

In [ ]:
# Chapter-style tree.
# A is MAX. B, C, D are MIN. Leaves are utilities for MAX.
TREE = {
    "A": {"type": "MAX", "children": ["B", "C", "D"]},
    "B": {"type": "MIN", "children": ["B1", "B2", "B3"]},
    "C": {"type": "MIN", "children": ["C1", "C2", "C3"]},
    "D": {"type": "MIN", "children": ["D1", "D2", "D3"]},
    "B1": {"type": "LEAF", "value": 3}, "B2": {"type": "LEAF", "value": 12}, "B3": {"type": "LEAF", "value": 8},
    "C1": {"type": "LEAF", "value": 2}, "C2": {"type": "LEAF", "value": 4}, "C3": {"type": "LEAF", "value": 6},
    "D1": {"type": "LEAF", "value": 14}, "D2": {"type": "LEAF", "value": 5}, "D3": {"type": "LEAF", "value": 2},
}

POS = {
    "A": (0, 0),
    "B": (-4, -1.5), "C": (0, -1.5), "D": (4, -1.5),
    "B1": (-5.1, -3), "B2": (-4, -3), "B3": (-2.9, -3),
    "C1": (-1.1, -3), "C2": (0, -3), "C3": (1.1, -3),
    "D1": (2.9, -3), "D2": (4, -3), "D3": (5.1, -3),
}

def descendants(node):
    todo = [node]
    found = []
    while todo:
        n = todo.pop()
        found.append(n)
        todo.extend(TREE[n].get("children", []))
    return found


def minimax_tree_value(node):
    info = TREE[node]
    if info["type"] == "LEAF":
        return info["value"]
    vals = [minimax_tree_value(child) for child in info["children"]]
    return max(vals) if info["type"] == "MAX" else min(vals)

print("Minimax value at root A:", minimax_tree_value("A"))

In [ ]:
def alpha_beta_trace(root="A"):
    events = []
    def search(node, alpha, beta):
        node_type = TREE[node]["type"]
        events.append({
            "kind": "enter", "node": node, "alpha": alpha, "beta": beta,
            "message": f"Enter {node} ({node_type}) with alpha={fmt_bound(alpha)}, beta={fmt_bound(beta)}."
        })
        if node_type == "LEAF":
            value = TREE[node]["value"]
            events.append({
                "kind": "leaf", "node": node, "value": value, "alpha": alpha, "beta": beta,
                "message": f"Evaluate leaf {node}: utility={value}."
            })
            return value
        if node_type == "MAX":
            value = -math.inf
            for i, child in enumerate(TREE[node]["children"]):
                child_value = search(child, alpha, beta)
                value = max(value, child_value)
                alpha = max(alpha, value)
                events.append({
                    "kind": "update", "node": node, "child": child, "value": value,
                    "alpha": alpha, "beta": beta,
                    "message": f"MAX node {node}: after {child}, best value={value}; alpha becomes {fmt_bound(alpha)}."
                })
                if value >= beta:
                    pruned = []
                    for remaining in TREE[node]["children"][i+1:]:
                        pruned.extend(descendants(remaining))
                    events.append({
                        "kind": "prune", "node": node, "pruned": pruned, "value": value,
                        "alpha": alpha, "beta": beta,
                        "message": f"Prune at MAX node {node}: value {value} >= beta {fmt_bound(beta)}."
                    })
                    break
            events.append({
                "kind": "return", "node": node, "value": value, "alpha": alpha, "beta": beta,
                "message": f"Return value {value} from MAX node {node}."
            })
            return value
        if node_type == "MIN":
            value = math.inf
            for i, child in enumerate(TREE[node]["children"]):
                child_value = search(child, alpha, beta)
                value = min(value, child_value)
                beta = min(beta, value)
                events.append({
                    "kind": "update", "node": node, "child": child, "value": value,
                    "alpha": alpha, "beta": beta,
                    "message": f"MIN node {node}: after {child}, best value={value}; beta becomes {fmt_bound(beta)}."
                })
                if value <= alpha:
                    pruned = []
                    for remaining in TREE[node]["children"][i+1:]:
                        pruned.extend(descendants(remaining))
                    events.append({
                        "kind": "prune", "node": node, "pruned": pruned, "value": value,
                        "alpha": alpha, "beta": beta,
                        "message": f"Prune at MIN node {node}: value {value} <= alpha {fmt_bound(alpha)}."
                    })
                    break
            events.append({
                "kind": "return", "node": node, "value": value, "alpha": alpha, "beta": beta,
                "message": f"Return value {value} from MIN node {node}."
            })
            return value
    root_value = search(root, -math.inf, math.inf)
    events.append({"kind": "done", "node": root, "value": root_value, "message": f"Done. Root value={root_value}."})
    return events

AB_EVENTS = alpha_beta_trace("A")
print("Alpha-beta events:", len(AB_EVENTS))
print("Leaves evaluated:", [e["node"] for e in AB_EVENTS if e["kind"] == "leaf"])

In [ ]:
def draw_ab_tree(ev, idx):
    prefix = AB_EVENTS[:idx+1]
    current = ev.get("node")
    evaluated = {e["node"] for e in prefix if e["kind"] == "leaf"}
    returned = {e["node"]: e["value"] for e in prefix if e["kind"] in ("leaf", "return", "done")}
    pruned = set()
    bounds = {}
    for e in prefix:
        if e["kind"] == "prune":
            pruned.update(e.get("pruned", []))
        if "alpha" in e and "beta" in e:
            bounds[e["node"]] = (e["alpha"], e["beta"])
    fig, ax = plt.subplots(figsize=(12, 6))
    for parent, info in TREE.items():
        for child in info.get("children", []):
            x1, y1 = POS[parent]
            x2, y2 = POS[child]
            if child in pruned:
                ax.plot([x1, x2], [y1, y2], ls="--", color="#bbbbbb", lw=1)
            else:
                ax.plot([x1, x2], [y1, y2], color="#444444", lw=1.5)
    for node, info in TREE.items():
        x, y = POS[node]
        if node in pruned:
            face, edge, lw = "#eeeeee", "#999999", 1
        elif node == current:
            face, edge, lw = "#ffbf66", "#111111", 2.5
        elif node in returned:
            face, edge, lw = "#b6e3a8", "#333333", 1.5
        elif node in evaluated:
            face, edge, lw = "#d7e9ff", "#333333", 1.3
        else:
            face, edge, lw = "#f7f7f7", "#666666", 1
        marker = "o"
        if info["type"] == "MAX": marker = "^"
        if info["type"] == "MIN": marker = "v"
        ax.scatter([x], [y], s=950 if info["type"] != "LEAF" else 750, marker=marker, c=face, edgecolors=edge, linewidths=lw, zorder=3)
        label = node
        if info["type"] == "LEAF":
            label += f"\n{info['value']}"
        if node in returned and info["type"] != "LEAF":
            label += f"\nv={returned[node]:g}"
        if node in bounds and info["type"] != "LEAF":
            a, b = bounds[node]
            label += f"\nα={fmt_bound(a)}, β={fmt_bound(b)}"
        ax.text(x, y, label, ha="center", va="center", fontsize=9, zorder=4)
        if node in pruned:
            ax.text(x, y-0.45, "pruned", ha="center", va="center", fontsize=8, color="#777777")
    ax.set_title("Alpha-beta search: evaluated, returned, and pruned nodes")
    ax.axis("off")
    ax.set_xlim(-6.2, 6.2)
    ax.set_ylim(-3.8, 0.8)
    plt.show()


def ab_info_html(ev, idx):
    return f"""
    <div style='font-family:Arial;line-height:1.35'>
    <b>Step {idx+1} of {len(AB_EVENTS)}</b> &nbsp; <b>Event:</b> {ev['kind']}<br>
    {ev['message']}<br>
    MAX nodes use the largest backed-up value; MIN nodes use the smallest. A dashed gray subtree has been pruned.
    </div>
    """

make_stepper(AB_EVENTS, draw_ab_tree, ab_info_html, title="Alpha-beta pruning stepper")

## Experiment: why move ordering matters

The next cells generate many random game trees. The leaf values are random utilities. We compare:

- plain minimax, which visits every node;
- alpha-beta with random ordering;
- alpha-beta with best possible ordering, which is not realistic but gives the theoretical target;
- alpha-beta with worst possible ordering.

In [ ]:
def random_tree(branching, depth, rng):
    if depth == 0:
        return rng.randint(-100, 100)
    return [random_tree(branching, depth-1, rng) for _ in range(branching)]


def true_value(tree, maximizing=True):
    if not isinstance(tree, list):
        return tree
    vals = [true_value(ch, not maximizing) for ch in tree]
    return max(vals) if maximizing else min(vals)


def minimax_count(tree, maximizing=True):
    if not isinstance(tree, list):
        return tree, 1
    values, count = [], 1
    for ch in tree:
        v, c = minimax_count(ch, not maximizing)
        values.append(v)
        count += c
    return (max(values) if maximizing else min(values)), count


def alphabeta_count(tree, alpha=-math.inf, beta=math.inf, maximizing=True, order="given"):
    if not isinstance(tree, list):
        return tree, 1
    children = list(tree)
    if order in {"best", "worst"}:
        reverse = maximizing
        children.sort(key=lambda ch: true_value(ch, not maximizing), reverse=reverse)
        if order == "worst":
            children.reverse()
    count = 1
    if maximizing:
        value = -math.inf
        for ch in children:
            v, c = alphabeta_count(ch, alpha, beta, False, order)
            count += c
            value = max(value, v)
            alpha = max(alpha, value)
            if value >= beta:
                break
        return value, count
    value = math.inf
    for ch in children:
        v, c = alphabeta_count(ch, alpha, beta, True, order)
        count += c
        value = min(value, v)
        beta = min(beta, value)
        if value <= alpha:
            break
    return value, count


def run_ordering_experiment(branching=3, max_depth=8, trials=40, seed=7):
    rng = random.Random(seed)
    rows = []
    for depth in range(1, max_depth+1):
        counts = defaultdict(list)
        for _ in range(trials):
            tree = random_tree(branching, depth, rng)
            _, c_minimax = minimax_count(tree)
            _, c_random = alphabeta_count(tree, order="given")
            _, c_best = alphabeta_count(tree, order="best")
            _, c_worst = alphabeta_count(tree, order="worst")
            counts["minimax"].append(c_minimax)
            counts["alpha_beta_random"].append(c_random)
            counts["alpha_beta_best"].append(c_best)
            counts["alpha_beta_worst"].append(c_worst)
        rows.append({name: mean(vals) for name, vals in counts.items()} | {"depth": depth})
    return rows

experiment_rows = run_ordering_experiment(branching=3, max_depth=8, trials=60)
for row in experiment_rows:
    print(row)

In [ ]:
def plot_ordering_experiment(rows):
    depths = [r["depth"] for r in rows]
    fig, ax = plt.subplots(figsize=(9, 5))
    for key, label in [
        ("minimax", "Minimax"),
        ("alpha_beta_random", "Alpha-beta random order"),
        ("alpha_beta_best", "Alpha-beta best order"),
        ("alpha_beta_worst", "Alpha-beta worst order"),
    ]:
        ax.plot(depths, [r[key] for r in rows], marker="o", label=label)
    ax.set_yscale("log")
    ax.set_xlabel("search depth m")
    ax.set_ylabel("mean nodes visited, log scale")
    ax.set_title("Move ordering changes how much alpha-beta can prune")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend()
    plt.show()

plot_ordering_experiment(experiment_rows)

## Student practice

Change the branching factor, search depth, and number of trials. Explain why best ordering is not available in a real game, and why heuristic ordering can still help.

In [ ]:
branching_factor = 4
maximum_depth = 6
trials = 25

student_rows = run_ordering_experiment(branching=branching_factor, max_depth=maximum_depth, trials=trials, seed=23)
plot_ordering_experiment(student_rows)